# Optimisation MILP du dimensionnement d'un système PV/batterie/réseau sous contrainte de couverture de charge (demande résidentielle), minimisation du coût total.  
# Résolution par Pyomo
Solveur: %pip install highspy

On doit répondre à la demande d'un foyer sur un an.  
On peut y répondre par des panneaux photovoltaiques, des batteries et un appel à un réseau. 

### **Calcul du productible solaire**  
%pip install pvlib

Le modèle considère un panneau photovoltaïque monocristallin de 425 Wc représentatif du marché français actuel (DualSun Flash 425 W). Son coût complet installé (module, fixation, pose, câblage et maintenance) est approché par 1000 €/panneau.  
Le lieu choisi pour mettre les panneaux est Grenoble.

In [ ]:
import pvlib
from pvlib.location import Location
from pvlib.modelchain import ModelChain
from pvlib.pvsystem import PVSystem, Array, FixedMount

#On veux les données pour grenoble
latitude = 45.1885
longitude = 5.7247

loc = Location(
    latitude,
    longitude,
    tz="Europe/Paris",
    altitude=100,
    name="Grenoble"
)

#On a les paramètres d'ensoleillement pour Grenoble
weather, meta = pvlib.iotools.get_pvgis_tmy(
    latitude=latitude,
    longitude=longitude,
    map_variables=True
)

weather.index = weather.index.tz_convert("Europe/Paris")

#Panneau PV
panneau_DualSun_Flash_425_W = { 
        "pdc0":425, #Puissance crète nominale à 100W/m² et Temp ambiante (25°)
        "gamma_pdc":-0.0031 #Pertes d'efficacité selon température (%/°C)
    }

#Onduleur
onduleur = {
        "pdc0":425,
        "eta_inv_nom":0.966, #Efficacité pour passer de Dc à AC
        "eta_inv_ref":0.9637
    }

#Paramètres du modèle thermique
temperature_model_parameters = (
    pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS[
        "sapm"
    ]["open_rack_glass_glass"]
)

#On crée notre système (PV + onduleur)
system = PVSystem(
    arrays=[
        Array(
            mount=FixedMount(
                surface_tilt=30,
                surface_azimuth=180
            ),
            module_parameters=panneau_DualSun_Flash_425_W,
            temperature_model_parameters=temperature_model_parameters,
            modules_per_string=1,
            strings=1
        )
    ],
    inverter_parameters=onduleur
)

#On compte les pertes
mc = ModelChain.with_pvwatts(system, loc)

#On calcule la production horaire du panneau
mc.run_model(weather)

#La production en kwh
production_kwh = mc.results.ac / 1000

#Le df utilisé par pyomo
df_production_1pv = production_kwh.reset_index(drop=True)

#La production annuelle
print(production_kwh.sum())


566.9837890975473


Caractéristiques économiques des panneaux solaires

In [ ]:
prix_panneau_pv_sans_pose = 300 #€/panneau sans pose
#Environ 600€ pour la pose
#Donc prix total = 1000€
prix_panneau_pv = 1000 #pose incluse

Caractéristiques technico-économiques des batteries

In [ ]:
prix_kw_batterie = 120
prix_kwh_batterie = 350
rendement_charge_batterie = 0.95
rendement_decharge_batterie = 0.95

# Discretisation : la batterie est dimensionnee comme un nombre entier de
# modules standards empilables (type BYD Battery-Box Premium / Enphase IQ),
# plutot qu'une puissance/capacite continue (irrealiste commercialement).
# Hypothese par defaut : module de 2.5 kWh / 1.25 kW (ratio de puissance 0.5C).
# -> A ajuster selon le produit reel vise.
capacite_module_batterie = 2.5   # kWh par module
puissance_module_batterie = 1.25  # kW par module


### **Récupération des prix de l'électricité (achat et vente)**    
%pip install entsoe-py

La clé API ENTSOE est récupérable en récupérant un compte ENTSOE sur la plateforme: https://transparency.entsoe.eu/

In [ ]:
import os
import getpass

# Demande la clé à l'utilisateur sans l'afficher à l'écran
api_key = getpass.getpass("Entrez votre clé API : ")

# Définit la variable d'environnement pour ce processus
os.environ["ENTSOE_API_KEY"] = api_key   

On étudie **deux contrats d'électricité** pour un particulier:  
-tarif fixe classique (heures pleines/creuses)    
-tarif dynamique (prix spot)


In [ ]:
#Contrat d'achat

#CONTRAT FIXE
abonnement_annuel= 15.65*12 #€/an
abonnement_jour = abonnement_annuel/365 #€/jour
abonnement_heure = abonnement_jour/24 #€/h

#Heures creuses : 22h-6h et 12h-14h
prix_energie_heure_creuse = 0.1579 #€/kWh
#Heures pleines : 6h-12h et 14h-22h
prix_energie_heure_pleine = 0.2065 #€/kWh

prix_jour = []
for h in range(24):
    if (h >= 22 or h < 6) or (h >= 12 and h < 14):
        prix_jour.append(prix_energie_heure_creuse )
    else:
        prix_jour.append(prix_energie_heure_pleine)

df_achat_elec_contrat_fixe = []
for jour in range (365):
    df_achat_elec_contrat_fixe.extend(prix_jour)

#CONTRAT SPOT
from entsoe import EntsoePandasClient
import pandas as pd
import os

def df_achat_elec_contrat_spot():

    # Cle API ENTSO-E : NE JAMAIS coder une cle en dur dans un notebook publie.
    # Definir la variable d'environnement ENTSOE_API_KEY avant de lancer Jupyter, ex (terminal) :
    #   export ENTSOE_API_KEY="ta_cle_ici"
    # ou via un fichier .env (non versionne / non commit) charge avec python-dotenv.
    api_key = os.environ.get("ENTSOE_API_KEY")
    if api_key is None:
        print("Cle API ENTSO-E introuvable (variable d'environnement ENTSOE_API_KEY absente). "
              "Le tarif spot ne sera pas calcule ; le notebook continue avec le contrat fixe.")
        return None

    client = EntsoePandasClient(api_key=api_key)

    # Definir la periode (1 an complet)
    # Le fuseau horaire est crucial (ex: 'Europe/Paris' ou 'UTC')
    start = pd.Timestamp('2025-01-01', tz='Europe/Paris')
    end = pd.Timestamp('2026-01-01', tz='Europe/Paris')

    df_prix = client.query_day_ahead_prices(country_code='FR', start=start, end=end)

    #On passe des €/MWh aux €/kWh
    df_prix = df_prix/1000

    #Les differents formats de donnees qu'on a : horaire, quart heure...
    print(df_prix.index.to_series().diff().value_counts())

    df_prix = df_prix.resample("1h").mean()

    # index 0...8759 pour Pyomo
    df_prix = df_prix.reset_index(drop=True)

    return df_prix[:8760]

df_achat_elec_contrat_spot = df_achat_elec_contrat_spot()
#print(len(df_achat_elec_contrat_spot))

#Contrat de vente: 11cts/kwh
df_vente_elec = pd.Series(
    0.011,
    index=range(8760)
)


0 days 00:15:00    8834
0 days 01:00:00    6551
0 days 00:30:00       1
Name: count, dtype: int64


### **Récupération de la demande électrique d'un foyer**  
On utilise les coefficients d'enedis.  
On normalise les valeurs (somme des coeffs sur l'année = 1) et on ventile la consommation annuelle selon les coeffs horaires

In [ ]:
import pandas as pd

# Fichier public Enedis (dataset "Courbes de charge - Profils Enedis" sur data.enedis.fr).
# Place ce CSV dans le meme dossier que ce notebook (ou adapte le chemin ci-dessous).
coeffs_enedis = "coefficients-des-profils.csv"

df_coeffs = pd.read_csv(coeffs_enedis,sep=",")

df_demande = df_coeffs[["HORODATE", "COEFFICIENT_AJUSTE"]].copy()
df_demande.columns = ["date", "coeffs"]
df_demande["date"] = pd.to_datetime(df_demande["date"], utc=True)
df_demande = df_demande.set_index("date") #mise en index de la colonne
df_demande = df_demande.resample("h").mean() #conversion en donnees horaires

# Choix d'une annee
df_demande = df_demande.loc["2025"]

#Normalisation
df_demande ["coeffs_normalise"] = df_demande["coeffs"] / df_demande["coeffs"].sum() #normalisation des coefficients
#On verifie qu'on a bien une somme de 1
#print(df_demande["coeffs_normalise"].sum())

#Fixer une conso annuelle
demande_annuelle_elec_kwh = 4500
#Ventilee par heure sur 1 an
df_demande["demande elec kwh"] = df_demande["coeffs_normalise"] * demande_annuelle_elec_kwh

#Le df a utiliser plus tard
demande = df_demande["demande elec kwh"]
#print(demande.head())
#print(demande.sum()) #verification de la somme des demandes
#print(demande.max()) #verification de la demande maximale


0.9999999999999999
date
2025-01-01 00:00:00+00:00    0.423369
2025-01-01 01:00:00+00:00    0.386017
2025-01-01 02:00:00+00:00    0.366681
2025-01-01 03:00:00+00:00    0.356703
2025-01-01 04:00:00+00:00    0.354084
Freq: h, Name: demande elec kwh, dtype: float64
4500.0
On a bien decoupe!
1.06021923401457


### **Annualisation du capex (Capital Recovery Factor)**  
Sans ce facteur, aucun investissement (panneau, batterie) ne serait rentable; on aurait intérêt à acheter l'électricité sur le réseau.  
On annualise alors les couts du capex avec un facteur de récupération du capital (CRF): répartit le cout d'achat sur la durée de vie de l'équipement 

In [ ]:
# --------------
# Annualisation du capex (Capital Recovery Factor)
# --------------

taux_actualisation = 0.04  # hypothese : 4%, taux d'actualisation pour un menage

duree_vie_pv = 25          # ans, duree de vie typique d'un panneau PV
duree_vie_batterie = 12    # ans, duree de vie typique d'une batterie Li-ion residentielle

def crf(taux, duree):
    return taux * (1 + taux)**duree / ((1 + taux)**duree - 1)

CRF_pv = crf(taux_actualisation, duree_vie_pv)
CRF_batterie = crf(taux_actualisation, duree_vie_batterie)

print(f"CRF PV : {CRF_pv:.4f} (soit {CRF_pv*100:.2f} % du capex paye chaque annee)")
print(f"CRF batterie : {CRF_batterie:.4f} (soit {CRF_batterie*100:.2f} % du capex paye chaque annee)")


CRF PV : 0.0640 (soit 6.40 % du capex paye chaque annee)
CRF batterie : 0.1066 (soit 10.66 % du capex paye chaque annee)


## Scenarios : quel est le "cout de la transition" pour ce foyer ?

Ce qu'on constate si on laisse le solveur choisir : appel majoritaire au réseau, achat de 2 panneaux pour compléter le réseau et 0 batterie.

Ce qu'on décide alors de faire:  
**Imposer comme contrainte un certain taux d'auto-consommation.**  
Cela permettra de voir le cout nécessaire pour réaliser la transition énergétique.  
Le solveur optimisera alors au cout minimal le mix énergétique (pv, batterie, réseau) pour répondre à la demande et respecter le taux d'autoconsommation.  


**A propos du champ `status`** : `optimal` signifie que HiGHS a trouve (a la tolerance
`mip_gap` pres) la meilleure solution ; les autres valeurs possibles sont notamment
`infeasible` (aucune solution ne satisfait toutes les contraintes -- verifie alors si
`taux_autoproduction_min` n'est pas trop eleve pour les bornes M actuelles) ou
`maxTimeLimit` (le solveur a ete arrete avant preuve d'optimalite).

In [ ]:
import pyomo.environ as pyo
import pandas as pd

def construire_et_resoudre_modele(df_achat_elec_prix, taux_autoproduction_min=None,
                                   autoriser_deconnexion_reseau=False, mip_gap=0.01):
    """
    Construit et resout le probleme de dimensionnement PV + batterie + reseau.

    Seules 2 variables sont entieres (nombre_panneaux_pv, nb_modules_batterie) :
    le reste est un programme lineaire (LP). On n'impose PAS explicitement
    l'exclusivite "charge OU decharge" / "achat OU vente" par des binaires+grand-M :
    ce n'est pas necessaire ici, car ces situations sont dominees economiquement
    (acheter a 0.16-0.21 €/kWh et vendre a 0.011 €/kWh la meme heure ferait perdre
    de l'argent pour rien ; charger et decharger la meme heure gaspillerait de
    l'energie via les rendements de charge/decharge de 95%). Le solveur ne choisira
    donc jamais ces cas spontanement. Supprimer ces ~17 500 binaires (8760h x 2)
    fait passer la resolution d'un gros MILP (potentiellement tres lent, gourmand
    en memoire) a une LP quasi-instantanee, sans changer le resultat optimal.

    df_achat_elec_prix : serie de prix d'achat horaires (€/kWh), longueur 8760.
    taux_autoproduction_min : float entre 0 et 1, ou None. Si fourni, impose que
        (pv_maison + batterie_maison) couvre au moins cette fraction de la
        demande annuelle.
    autoriser_deconnexion_reseau : si True, un binaire rend l'abonnement reseau
        optionnel (un seul binaire ajoute, sans impact notable sur le temps de
        resolution).
    mip_gap : tolerance d'optimalite relative pour HiGHS.

    Retourne un dict avec les resultats, la decomposition du cout par poste,
    et le modele Pyomo resolu (pour extraire des series horaires si besoin).
    """

    model = pyo.ConcreteModel()
    model.time = pyo.RangeSet(1, 8760)

    # --------------
    #Parametres du modele
    # --------------

    model.prix_panneau_pv = pyo.Param(initialize=1000)
    model.prix_kw_batterie = pyo.Param(initialize=prix_kw_batterie)
    model.prix_kwh_batterie = pyo.Param(initialize=prix_kwh_batterie)
    model.rendement_charge_batterie = pyo.Param(initialize=rendement_charge_batterie)
    model.rendement_decharge_batterie = pyo.Param(initialize=rendement_decharge_batterie)
    model.abonnement_annuel_elec = pyo.Param(initialize=abonnement_annuel)
    model.CRF_pv = pyo.Param(initialize=CRF_pv)
    model.CRF_batterie = pyo.Param(initialize=CRF_batterie)
    model.capacite_module_batterie = pyo.Param(initialize=capacite_module_batterie)
    model.puissance_module_batterie = pyo.Param(initialize=puissance_module_batterie)

    df_achat_elec_serie = pd.Series(df_achat_elec_prix).reset_index(drop=True)
    model.prix_achat_electricite = pyo.Param(model.time, domain=pyo.Reals, initialize=lambda model, t: df_achat_elec_serie.iloc[t-1])

    df_vente_elec_serie = pd.Series(df_vente_elec).reset_index(drop=True)
    model.prix_vente_electricite = pyo.Param(model.time, domain=pyo.Reals, initialize=lambda model, t: df_vente_elec_serie.iloc[t-1])

    df_demande = demande.reset_index(drop=True)
    model.demande_electricite = pyo.Param(model.time, domain=pyo.NonNegativeReals, initialize=lambda model, t: df_demande.iloc[t-1])

    df_production_1pv = production_kwh.reset_index(drop=True)
    model.production_1pv = pyo.Param(model.time, domain=pyo.NonNegativeReals, initialize=lambda model, t: df_production_1pv.iloc[t-1])

    # --------------
    #Variables d'optimisation
    # --------------

    model.nombre_panneaux_pv = pyo.Var(domain=pyo.NonNegativeIntegers)
    model.nb_modules_batterie = pyo.Var(domain=pyo.NonNegativeIntegers)
    model.pmax_batterie = pyo.Var(domain=pyo.NonNegativeReals)
    model.estock_max_batterie = pyo.Var(domain=pyo.NonNegativeReals)

    model.achat_electricite = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.vente_electricite = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.pv_maison = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.pv_batterie = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.pv_grid = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.grid_batterie = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.grid_maison = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.batterie_maison = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.batterie_grid = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.echarge = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.edecharge = pyo.Var(model.time, domain=pyo.NonNegativeReals)
    model.estock_batterie = pyo.Var(model.time, domain=pyo.NonNegativeReals)

    #Connexion reseau optionnelle (par defaut : toujours connecte -> Param fixe a 1)
    if autoriser_deconnexion_reseau:
        model.connecte_reseau = pyo.Var(domain=pyo.Binary)
    else:
        model.connecte_reseau = pyo.Param(initialize=1)

    # --------------
    #Fonction objectif (capex annualise via CRF + opex annuel)
    # --------------

    model.obj = pyo.Objective(expr = model.abonnement_annuel_elec * model.connecte_reseau +
                            model.nombre_panneaux_pv * model.prix_panneau_pv * model.CRF_pv +
                            model.pmax_batterie * model.prix_kw_batterie * model.CRF_batterie +
                            model.estock_max_batterie * model.prix_kwh_batterie * model.CRF_batterie +
                            sum(model.achat_electricite[t] * model.prix_achat_electricite[t]
                                - model.vente_electricite[t] * model.prix_vente_electricite[t]
                                for t in model.time), sense=pyo.minimize)

    # --------------
    # Contraintes
    # --------------

    def rule_bilan_pv(model, t):
        return model.production_1pv[t] * model.nombre_panneaux_pv == \
               model.pv_maison[t] + model.pv_batterie[t] + model.pv_grid[t]
    model.bilan_pv = pyo.Constraint(model.time, rule=rule_bilan_pv)

    def rule_bilan_grid_achat(model, t):
        return model.achat_electricite[t] == model.grid_batterie[t] + model.grid_maison[t]
    model.bilan_grid_achat = pyo.Constraint(model.time, rule=rule_bilan_grid_achat)

    def rule_bilan_grid_vente(model, t):
        return model.vente_electricite[t] == model.batterie_grid[t] + model.pv_grid[t]
    model.bilan_grid_vente = pyo.Constraint(model.time, rule=rule_bilan_grid_vente)

    def rule_echargement_batterie(model, t):
        return model.grid_batterie[t] + model.pv_batterie[t] == model.echarge[t]
    model.echargement_batterie = pyo.Constraint(model.time, rule=rule_echargement_batterie)

    def rule_dechargement_batterie(model, t):
        return model.batterie_maison[t] + model.batterie_grid[t] == model.edecharge[t]
    model.dechargement_batterie = pyo.Constraint(model.time, rule=rule_dechargement_batterie)

    def rule_estock_batterie(model, t):
        if t > 1:
            return model.estock_batterie[t] == model.estock_batterie[t-1] + model.echarge[t]*model.rendement_charge_batterie - model.edecharge[t]/model.rendement_decharge_batterie
        else:
            return pyo.Constraint.Skip
    model.estock_batterie_contrainte = pyo.Constraint(model.time, rule=rule_estock_batterie)

    model.estock_batterie_initial = pyo.Constraint(rule=lambda model: model.estock_batterie[1] == 0)
    model.estock_batterie_final = pyo.Constraint(rule=lambda model: model.estock_batterie[8760] == 0)

    model.e_min = pyo.Constraint(model.time, rule=lambda model, t: model.estock_batterie[t] >= 0)
    model.e_max = pyo.Constraint(model.time, rule=lambda model, t: model.estock_batterie[t] <= model.estock_max_batterie)

    #Puissance max de charge/decharge (limite physique reelle de la batterie)
    model.pmax_charge = pyo.Constraint(model.time, rule=lambda model, t: model.echarge[t] <= model.pmax_batterie)
    model.pmax_decharge = pyo.Constraint(model.time, rule=lambda model, t: model.edecharge[t] <= model.pmax_batterie)

    model.lien_capacite_batterie = pyo.Constraint(rule=lambda model: model.estock_max_batterie == model.nb_modules_batterie * model.capacite_module_batterie)
    model.lien_puissance_batterie = pyo.Constraint(rule=lambda model: model.pmax_batterie == model.nb_modules_batterie * model.puissance_module_batterie)

    #NB : pas de contrainte d'exclusivite charge/decharge ni achat/vente -- voir
    #docstring : ces cas sont domines economiquement, jamais choisis par le solveur.

    def rule_bilan_maison(model, t):
        return model.demande_electricite[t] == model.pv_maison[t] + model.grid_maison[t] + model.batterie_maison[t]
    model.bilan_maison = pyo.Constraint(model.time, rule=rule_bilan_maison)

    if taux_autoproduction_min is not None:
        def rule_taux_autoproduction(model):
            return sum(model.pv_maison[t] + model.batterie_maison[t] for t in model.time) >= \
                   taux_autoproduction_min * sum(model.demande_electricite[t] for t in model.time)
        model.contrainte_autoproduction = pyo.Constraint(rule=rule_taux_autoproduction)

    # --------------
    # Resolution
    # --------------

    solver = pyo.SolverFactory("highs")
    solver.options["mip_rel_gap"] = mip_gap
    result = solver.solve(model)

    # --------------
    # Decomposition du cout total par poste
    # --------------
    cout_abonnement = pyo.value(model.abonnement_annuel_elec * model.connecte_reseau)
    cout_capex_pv = pyo.value(model.nombre_panneaux_pv * model.prix_panneau_pv * model.CRF_pv)
    cout_capex_batterie = pyo.value((model.pmax_batterie * model.prix_kw_batterie + model.estock_max_batterie * model.prix_kwh_batterie) * model.CRF_batterie)
    cout_net_energie = pyo.value(sum(model.achat_electricite[t] * model.prix_achat_electricite[t]
                                      - model.vente_electricite[t] * model.prix_vente_electricite[t]
                                      for t in model.time))

    return {
        "taux_autoproduction_min": taux_autoproduction_min if taux_autoproduction_min is not None else 0.0,
        "status": str(result.solver.termination_condition),
        "n_pv": pyo.value(model.nombre_panneaux_pv),
        "n_batt": pyo.value(model.nb_modules_batterie),
        "pmax_batterie_kw": pyo.value(model.pmax_batterie),
        "emax_batterie_kwh": pyo.value(model.estock_max_batterie),
        "connecte_reseau": pyo.value(model.connecte_reseau),
        "cout_annualise_eur": pyo.value(model.obj),
        "cout_abonnement_eur": cout_abonnement,
        "cout_capex_pv_eur": cout_capex_pv,
        "cout_capex_batterie_eur": cout_capex_batterie,
        "cout_net_energie_eur": cout_net_energie,
        "model": model,
    }


### **Contrat fixe**
On souhaite observer les prix du système selon le taux d'autoconsommation

In [ ]:
# Balayage de scenarios de taux d'autoproduction impose (contrat fixe)
taux_scenarios = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

resultats_scenarios = []
for taux in taux_scenarios:
    r = construire_et_resoudre_modele(df_achat_elec_contrat_fixe, taux_autoproduction_min=taux) #Renvoie un dictionnaire
    resultats_scenarios.append(r)
    print(f"Taux autoproduction >= {taux:.0%} -> "
          f"{r['n_pv']:.0f} panneaux, {r['n_batt']:.0f} modules batterie, "
          f"cout annualise = {r['cout_annualise_eur']:.0f} €/an  ({r['status']})")

df_scenarios = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in resultats_scenarios])
df_scenarios["surcout_vs_optimum_libre_eur"] = df_scenarios["cout_annualise_eur"] - df_scenarios["cout_annualise_eur"].iloc[0]
df_scenarios


Taux autoproduction >= 0% -> 2 panneaux, 0 modules batterie, cout annualise = 970 €/an  (optimal)
Taux autoproduction >= 20% -> 2 panneaux, 0 modules batterie, cout annualise = 970 €/an  (optimal)


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

ax1.plot(df_scenarios["taux_autoproduction_min"]*100, df_scenarios["cout_annualise_eur"], marker="o", color="tab:blue")
ax1.set_xlabel("Taux d'autoproduction impose (%)")
ax1.set_ylabel("Cout annualise total (€/an)", color="tab:blue")
ax1.tick_params(axis='y', labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.bar(df_scenarios["taux_autoproduction_min"]*100, df_scenarios["n_pv"], width=4, alpha=0.3, color="tab:orange", label="Nb panneaux")
ax2.set_ylabel("Nombre de panneaux PV", color="tab:orange")
ax2.tick_params(axis='y', labelcolor="tab:orange")

plt.title("Cout de la transition energetique du foyer selon le taux d'autoproduction impose")
fig.tight_layout()
plt.show()


**Lecture du resultat** : la premiere ligne (0 %) correspond a l'optimum de marche pur -- souvent
"tout au reseau" si les prix actuels sont bas. Les lignes suivantes montrent combien il faut payer
en plus (par an) pour imposer un mix PV + batterie plus consequent, et quelle est la combinaison
la moins chere pour y arriver a chaque palier. C'est cette courbe qui permet de discuter du
"cout de la transition" plutot que de conclure un peu vite que le PV/la batterie "ne sert a rien".

*Remarque technique* : au-dela de 80-100 % d'autoproduction imposee, le nombre de panneaux/modules
necessaires pour couvrir les pics d'hiver peut devenir tres grand (et le temps de resolution du
MILP augmenter en consequence) -- c'est attendu, l'autoproduction totale d'une maison sans
soutien du reseau est intrinsequement couteuse.


Contrat spot

In [ ]:
# Balayage de scenarios de taux d'autoproduction impose (contrat spot)
taux_scenarios = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

resultats_scenarios = []
for taux in taux_scenarios:
    r = construire_et_resoudre_modele(df_achat_elec_contrat_spot, taux_autoproduction_min=taux)
    resultats_scenarios.append(r)
    print(f"Taux autoproduction >= {taux:.0%} -> "
          f"{r['n_pv']:.0f} panneaux, {r['n_batt']:.0f} modules batterie, "
          f"cout annualise = {r['cout_annualise_eur']:.0f} €/an  ({r['status']})")

df_scenarios = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in resultats_scenarios])
df_scenarios["surcout_vs_optimum_libre_eur"] = df_scenarios["cout_annualise_eur"] - df_scenarios["cout_annualise_eur"].iloc[0]
df_scenarios

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

ax1.plot(df_scenarios["taux_autoproduction_min"]*100, df_scenarios["cout_annualise_eur"], marker="o", color="tab:blue")
ax1.set_xlabel("Taux d'autoproduction impose (%)")
ax1.set_ylabel("Cout annualise total (€/an)", color="tab:blue")
ax1.tick_params(axis='y', labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.bar(df_scenarios["taux_autoproduction_min"]*100, df_scenarios["n_pv"], width=4, alpha=0.3, color="tab:orange", label="Nb panneaux")
ax2.set_ylabel("Nombre de panneaux PV", color="tab:orange")
ax2.tick_params(axis='y', labelcolor="tab:orange")

plt.title("Cout de la transition energetique du foyer selon le taux d'autoproduction impose")
fig.tight_layout()
plt.show()

## Decomposition du cout par poste, et acces aux series horaires

Le dict retourne par `construire_et_resoudre_modele` contient maintenant la decomposition
du cout annualise (`cout_abonnement_eur`, `cout_capex_pv_eur`, `cout_capex_batterie_eur`,
`cout_net_energie_eur`) et le modele Pyomo resolu lui-meme (`model`), reutilisable pour
extraire n'importe quelle serie horaire apres coup.


In [ ]:
# Stacked bar chart de la decomposition du cout par poste, pour chaque scenario
import matplotlib.pyplot as plt

postes = ["cout_abonnement_eur", "cout_capex_pv_eur", "cout_capex_batterie_eur", "cout_net_energie_eur"]
labels = ["Abonnement", "Capex PV (annualise)", "Capex batterie (annualise)", "Energie nette (achat-vente)"]

fig, ax = plt.subplots()
bottom = [0] * len(df_scenarios)
for poste, label in zip(postes, labels):
    ax.bar(df_scenarios["taux_autoproduction_min"]*100, df_scenarios[poste], bottom=bottom, width=8, label=label)
    bottom = [b + v for b, v in zip(bottom, df_scenarios[poste])]

ax.set_xlabel("Taux d'autoproduction impose (%)")
ax.set_ylabel("Cout annualise (€/an)")
ax.set_title("Composition du cout total selon le taux d'autoproduction impose")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Exemple : extraire les series horaires d'UN scenario precis pour un graphe detaille
# (ex: le scenario a taux impose = 60%)
r_exemple = resultats_scenarios[3]  # index 3 -> taux_scenarios[3] = 0.6
m = r_exemple["model"]

pv_maison = [pyo.value(m.pv_maison[t]) for t in m.time]
grid_maison = [pyo.value(m.grid_maison[t]) for t in m.time]
batterie_maison = [pyo.value(m.batterie_maison[t]) for t in m.time]
estock = [pyo.value(m.estock_batterie[t]) for t in m.time]

# Zoom sur une semaine (ex: la premiere semaine de janvier, heures 1 a 168)
h_debut, h_fin = 1, 168

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.stackplot(range(h_debut, h_fin+1),
              pv_maison[h_debut-1:h_fin], batterie_maison[h_debut-1:h_fin], grid_maison[h_debut-1:h_fin],
              labels=["PV -> maison", "Batterie -> maison", "Reseau -> maison"])
ax1.set_ylabel("kWh/h")
ax1.set_title(f"Couverture de la demande - premiere semaine (scenario taux={r_exemple['taux_autoproduction_min']:.0%})")
ax1.legend(loc="upper right")

ax2.plot(range(h_debut, h_fin+1), estock[h_debut-1:h_fin], color="tab:green")
ax2.set_ylabel("Etat de charge batterie (kWh)")
ax2.set_xlabel("Heure de l'annee")

plt.tight_layout()
plt.show()
